In [1]:
import pandas as pd
import os

def deduplicate_excel_column(input_file, column_name, output_file=None):
    """
    对Excel文件中指定列进行去重，保留第一次出现的行。
    
    参数:
        input_file (str): 输入的Excel文件路径
        column_name (str): 需要去重的列名
        output_file (str, optional): 输出文件路径。若为None，则自动生成新文件名
    """
    # 检查输入文件是否存在
    if not os.path.exists(input_file):
        print(f"错误：文件 '{input_file}' 不存在。")
        return

    # 读取Excel文件
    try:
        df = pd.read_excel(input_file, engine='openpyxl')
        print(f"成功读取文件，共 {len(df)} 行。")
    except Exception as e:
        print(f"读取文件时出错: {e}")
        return

    # 检查列是否存在
    if column_name not in df.columns:
        print(f"错误：列 '{column_name}' 不存在于文件中。可用列：{list(df.columns)}")
        return

    # 去重：基于指定列，保留第一次出现的行
    original_len = len(df)
    df_deduplicated = df.drop_duplicates(subset=[column_name], keep='first')
    new_len = len(df_deduplicated)

    print(f"去重完成：原行数 {original_len}，去重后行数 {new_len}，移除 {original_len - new_len} 行。")

    # 确定输出文件名
    if output_file is None:
        base, ext = os.path.splitext(input_file)
        output_file = f"{base}_deduplicated{ext}"

    # 保存结果
    try:
        df_deduplicated.to_excel(output_file, index=False, engine='openpyxl')
        print(f"去重后的数据已保存至：{output_file}")
    except Exception as e:
        print(f"保存文件时出错: {e}")

if __name__ == "__main__":
    # 使用示例
    input_filename = "news_output_classified1.xlsx"
    target_column = "alltext"
    deduplicate_excel_column(input_filename, target_column)

成功读取文件，共 27091 行。
去重完成：原行数 27091，去重后行数 19353，移除 7738 行。
去重后的数据已保存至：news_output_classified1_deduplicated.xlsx


In [6]:
import pandas as pd

# 文件路径
file1 = "gold_standard.xlsx"
file2 = "gold_standard_predicted1.csv"

# 需要对比的列名
compare_cols = [
    "G1", "G2", "G3", "L1", "L2", "L3",
    "R1", "R2", "R3", "C1", "C2", "C3", "C4", "C5",
    "OT", "NS"
]

# 读取文件
df1 = pd.read_excel(file1)
df2 = pd.read_csv(file2)

# 检查必要列
required_cols = ["样本ID"] + compare_cols
for col in required_cols:
    if col not in df1.columns:
        raise ValueError(f"{file1} 中缺少列 '{col}'")
    if col not in df2.columns:
        raise ValueError(f"{file2} 中缺少列 '{col}'")

# 统一样本ID为字符串
df1["样本ID"] = df1["样本ID"].astype(str)
df2["样本ID"] = df2["样本ID"].astype(str)

# 设为索引便于对齐
df1.set_index("样本ID", inplace=True)
df2.set_index("样本ID", inplace=True)

# 取共同ID
common_ids = set(df1.index) & set(df2.index)

# 存储每个ID的不一致信息
inconsistent_summary = []

for sid in common_ids:
    row1 = df1.loc[sid]
    row2 = df2.loc[sid]
    diff_cols = []
    for col in compare_cols:
        v1 = row1[col]
        v2 = row2[col]
        # 转换为整数比较（处理1.0、"1"等）
        try:
            v1 = int(v1) if pd.notna(v1) else None
            v2 = int(v2) if pd.notna(v2) else None
        except (ValueError, TypeError):
            pass
        if v1 != v2:
            diff_cols.append(col)
    if diff_cols:
        inconsistent_summary.append({
            "样本ID": sid,
            "不一致的列": ", ".join(diff_cols),
            "不一致列数": len(diff_cols)
        })

# 输出结果
if inconsistent_summary:
    df_summary = pd.DataFrame(inconsistent_summary)
    print(f"发现 {len(df_summary)} 个样本ID存在不一致：\n")
    print(df_summary.to_string(index=False))
    # 保存到CSV
    df_summary.to_csv("inconsistent_ids_summary.csv", index=False, encoding="utf-8-sig")
    print("\n汇总结果已保存至 inconsistent_ids_summary.csv")
else:
    print("所有共同样本ID的所有标签值完全一致！")

发现 168 个样本ID存在不一致：

样本ID                                                          不一致的列  不一致列数
 424                                                         G1, NS      2
 158                                                         R2, C1      2
 428                                                     G3, R1, R3      3
 164                                                         R1, NS      2
 408                                                         G1, NS      2
 163                                                             NS      1
  38                                                         R1, C1      2
 280                                                     L1, C1, NS      3
 440                                                         L2, L3      2
 399                                                             R3      1
 347                                                             NS      1
 370                                                             NS      1
  15 